In [ ]:
!pip install -q langchain langchain-community langchain-core chromadb sentence-transformers pypdf python-docx pandas groq openpyxl

In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

In [3]:
from google.colab import files
import pandas as pd
from docx import Document
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

text = ""

if file_name.endswith(".pdf"):
    loader = PyPDFLoader(file_name)
    documents = loader.load()
    text = " ".join([doc.page_content for doc in documents])

elif file_name.endswith(".docx"):
    doc = Document(file_name)
    text = " ".join([para.text for para in doc.paragraphs])

elif file_name.endswith(".csv"):
    df = pd.read_csv(file_name)
    text = df.to_string()

elif file_name.endswith(".xlsx"):
    df = pd.read_excel(file_name)
    text = df.to_string()

print("Text extraction completed ✅")

/tmp/ipykernel_6555/485497738.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Saving SWS-AI-benefits-compensation.pdf to SWS-AI-benefits-compensation.pdf
Text extraction completed ✅


In [4]:
import sys
!{sys.executable} -m pip install langchain-community

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.create_documents([text])

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Ensure directory exists
if not os.path.exists('./chroma_db'):
    os.makedirs('./chroma_db')

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma with persist_directory
vectorstore = Chroma.from_documents(
    docs,
    embedding,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever()
print("Vector store initialized and persisted to ./chroma_db ✅")

/tmp/ipykernel_6555/452743777.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store initialized and persisted to ./chroma_db ✅


In [7]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_llm(question, context):
    prompt = f"""
    Based ONLY on the following context, answer the question below.
    If the answer cannot be found in the context, please state that clearly.

    Context:
    {context}

    Question: {question}
    Answer:"""

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    return completion.choices[0].message.content

In [8]:
query = input("What would you like to ask about the document? ")
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])

answer = ask_llm(query, context)
print(answer)

What would you like to ask about the document? give relevant context about this
The context provided is about the benefits and compensation package offered by SWS AI to its employees. It includes details about:

1. Health Insurance: Coverage of INR 5,00,000 per annum for full-time employees, spouses, and up to 2 dependent children, with the premium fully borne by SWS AI.
2. Provident Fund: SWS AI contributes 12% of Basic Salary to the Employee Provident Fund, with employees contributing an equal 12%.
3. Learning & Development: Annual L&D budget of INR 20,000 per employee for courses, certifications, books, and conferences, with additional budgets for different role levels (INR 15,000, INR 25,000, and INR 40,000).
4. Internet & Equipment: Company-issued laptop, one-time equipment allowance of INR 10,000, but no internet reimbursement.
5. Leave Encashment: No details provided.
6. Stock Options: Eligibility for Senior engineers and above, based on performance and tenure.
7. Referral Bonus

In [ ]:
import shutil
import os
from google.colab import files

if os.path.exists('./chroma_db'):
    # Zip the chroma_db directory
    shutil.make_archive('chroma_db_backup', 'zip', './chroma_db')
    # Download the zipped database
    files.download('chroma_db_backup.zip')
    print('Vector store backup created and download started! ✅')
else:
    print('Error: ./chroma_db directory not found. Please run the vector store creation cell first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Vector store backup created and download started! ✅


### Step 1: Install Dependencies
We begin by installing the necessary libraries for LangChain, document processing, and vector storage.

### Step 2: Configure API Keys
Set up your Groq API key as an environment variable to allow the LLM to process queries.

### Step 3: Document Loading & Extraction
Upload your files (PDF, DOCX, CSV, XLSX) and extract the raw text content for processing.

### Step 4: Text Splitting
Divide the large extracted text into smaller, overlapping chunks to ensure the LLM stays within its context window.

### Step 5: Vector Store Initialization
Convert text chunks into numerical embeddings using HuggingFace and store them in a Chroma vector database for fast retrieval.

### Step 6: Querying the LLM
Define the retrieval logic and prompt the LLM to answer questions based strictly on the retrieved document context.

### Step 7: Backup and Download
Compress the persistent vector database into a ZIP file and download it for local use or future sessions.

# Task
Download the current Google Colab notebook as a `.ipynb` file. This is typically done via `File > Download > Download .ipynb` in the Colab interface.

## Download Current Notebook

### Subtask:
Instruct the user to download the current Google Colab notebook as a `.ipynb` file.


Please download the current Google Colab notebook as a `.ipynb` file by following these steps:

1.  **Navigate to 'File'**: In the top-left corner of the Google Colab interface, click on the `File` menu.
2.  **Select 'Download'**: From the dropdown menu, hover over `Download`.
3.  **Choose 'Download .ipynb'**: Click on `Download .ipynb` to save the notebook to your local machine.

Once the download is complete, please confirm that you have successfully saved the notebook.

## Final Task

### Subtask:
Confirm that the user has successfully downloaded the notebook in `.ipynb` format.
